# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
Dataset Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

**Note:** In this notebook, we reference all entities (record sets, fields, columns) by their Croissant `@id`.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset and its metadata using `mlcroissant`. The `Dataset` object provides access to the metadata and records as defined by the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Croissant Metadata object

print(f"{metadata.name}: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
List available record sets, their `@id`, and the fields (columns) in each record set. All references use the original Croissant `@id` for clarity and reproducibility.

In [ ]:
# Discover available record sets and their field `@id`s.
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the metadata.")
else:
    print("Available Record Sets and their fields (by @id):\n")
    for rs in record_sets:
        print(f"Record Set Name: {rs.name}")
        print(f"  @id: {rs.id}")
        if rs.fields:
            print("  Fields:")
            for f in rs.fields:
                print(f"    - {f.name} (@id: {f.id}) [dataType: {f.data_type}]")
        else:
            print("  No fields defined.")
        print("-")
# Store available record set @id's for later
record_set_ids = [rs.id for rs in record_sets]
# If you know the @id, you can use it directly. For demonstration, we take the first if exists.
if record_set_ids:
    example_record_set_id = record_set_ids[0]
else:
    example_record_set_id = None

## 3. Data Extraction
Let's load data from each record set into a pandas DataFrame for further analysis.

All dataset entities (record set and field names) are referenced by their Croissant `@id`.

In [ ]:
dataframes = {}
for rsid in record_set_ids:
    print(f"Loading records from record set @id: {rsid}")
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} records. Columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"Could not load record set {rsid}: {e}\n")

if example_record_set_id and example_record_set_id in dataframes:
    print(f"Example DataFrame columns for record set @id '{example_record_set_id}':")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No records loaded to DataFrame. Check the Croissant schema or try a different record set.")

## 4. Exploratory Data Analysis (EDA)
Let's apply some basic data processing to one of the loaded record sets (using its Croissant `@id`). We will select a numeric field by its `@id`, filter the records, normalize the values, and group by a key attribute if possible.

In [ ]:
# Choose a record set and numeric field for demonstration
# (If you know the @id, set these variables directly)
record_set_id = example_record_set_id

df = dataframes.get(record_set_id)
if df is None or df.empty:
    print("No data available for EDA in the selected record set.")
else:
    # Try to auto-detect a candidate numeric field by Croissant @id (field name)
    numeric_field_id = None
    for col in df.columns:
        # Look for common numeric terms, can be adjusted based on schema
        if any(k in col.lower() for k in ['num', 'count', 'total', 'value', 'coef', 'log', 'se', 'pval', 'error', 'score', 'mean']):
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    if numeric_field_id is None:
        # If not found, try to select the first numeric column
        for col in df.select_dtypes(include='number').columns:
            numeric_field_id = col
            break

    if numeric_field_id is None:
        print("No obvious numeric field found for EDA. Please review field definitions.")
    else:
        print(f"Selected numeric field (@id or name): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered rows where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        
        # Normalize numeric field
        norm_col_name = f"{numeric_field_id}_normalized"
        filtered_df[norm_col_name] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col_name]].head())
        
        # Group by a likely categorical field
        group_field = None
        for col in df.columns:
            # Guess a categorical/grouping field (can be improved per schema)
            if any(k in col.lower() for k in ['ward', 'county', 'id', 'gender', 'group', 'type', 'category']):
                if pd.api.types.is_string_dtype(df[col]):
                    group_field = col
                    break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} grouped by {group_field}:")
            print(grouped_df.head())
        else:
            print("No obvious grouping field found in this record set.")

## 5. Visualization
We will visualize the distribution of the selected numeric field and its normalized values, as well as (if available) a bar plot grouped by the categorical variable. All fields are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(filtered_df[numeric_field_id], kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.subplot(1, 2, 2)
    if norm_col_name in filtered_df:
        sns.histplot(filtered_df[norm_col_name], kde=True, color='lightgreen')
        plt.title(f"Normalized {numeric_field_id}")
        plt.xlabel(norm_col_name)
    plt.tight_layout()
    plt.show()

    # Bar plot by group field
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8, 4))
        sns.barplot(
            data=filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index(),
            x=group_field, y=numeric_field_id, palette='muted'
        )
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print("No filtered data available for visualization.")

## 6. Conclusion

In this notebook, we've demonstrated how to load, explore, and perform basic analysis on the FAIR^2 dataset using the `mlcroissant` library, referencing all dataset entities by their Croissant `@id`. You can extend this workflow with advanced statistical or ML methods, integrations, and visualizations as needed for data-driven projects in policy, intervention planning, or research.